In [1]:
import numpy as np

In [2]:
def conv1d(x, w, p=0, s=1):

    wrot = np.array(w[::-1])
    x_padded = np.array(x)
    if p > 0:
        zero_pad = np.zeros(shape=p)
        x_padded = np.concatenate(
            [zero_pad, x_padded, zero_pad]
        )

    res = []

    for i in range(0, int((len(x_padded) - len(wrot))) + 1, s):
        res.append(np.sum(x_padded[i:i+wrot.shape[0]]*[wrot]))

    return np.array(res)


x = [1, 3, 2, 4, 5, 6, 1, 3]
w = [1, 0, 3, 1, 2]

print(f'Conv1d implementation: {conv1d(x, w, p=2, s=1)}')

print(f'Numpy results: {np.convolve(x, w, mode='same')}')

Conv1d implementation: [ 5. 14. 16. 26. 24. 34. 19. 22.]
Numpy results: [ 5 14 16 26 24 34 19 22]


In [3]:
import scipy.signal

In [4]:
def conv2d(X, W, p=(0, 0), s=(1, 1)):

    W_rot = np.array(W)[::-1, ::-1]
    X_orig = np.array(X)

    n1 = X_orig.shape[0] + 2 * p[0]
    n2 = X_orig.shape[1] + 2 * p[1]

    X_padded = np.zeros(shape=(n1, n2))
    X_padded[
        p[0]:p[0] + X_orig.shape[0],
        p[1]:p[1] + X_orig.shape[1]
    ] = X_orig

    res = []
    for i in range(
        0,
        int((X_padded.shape[0] - W_rot.shape[0]) / s[0]) + 1,
        s[0],
    ):
        res.append([])

        for j in range(
            0,
            int((X_padded.shape[1] - W_rot.shape[1]) / s[1]) + 1,
            s[1],
        ):
            X_sub = X_padded[
                i:i + W_rot.shape[0],
                j:j + W_rot.shape[1],
            ]
            res[-1].append(np.sum(X_sub * W_rot))

    return np.array(res)

In [5]:
X = [[1, 3, 2, 4], [5, 6, 1, 3], [1, 2, 0, 2], [3, 4, 3, 2]]
W = [[1, 0, 3], [1, 2, 1], [0, 1, 1]]
print(f'Con2d Implementation: {conv2d(X, W, p=(1, 1), s=(1, 1))}')

print(scipy.signal.convolve2d(X, W, mode='same'))

Con2d Implementation: [[11. 25. 32. 13.]
 [19. 25. 24. 13.]
 [13. 28. 25. 17.]
 [11. 17. 14.  9.]]
[[11 25 32 13]
 [19 25 24 13]
 [13 28 25 17]
 [11 17 14  9]]


In [6]:
import torch 
from torchvision.io import read_image

In [8]:
img = read_image('example-image.png')
print(f'Image shape: {img.shape}')

Image shape: torch.Size([3, 252, 221])


In [9]:
import torch.nn as nn

In [11]:
loss_fn = nn.BCELoss()
loss = loss_fn(torch.tensor([0.9]), torch.tensor([1.0]))
l2_lambda = 0.001
conv_layer = nn.Conv2d(in_channels=3, out_channels=5, kernel_size=5)

l2_penalty = l2_lambda * sum(
    [(p**2).sum() for p in conv_layer.parameters()]
)

loss_with_penalty = loss + l2_penalty

print(f'Loss: {loss.item():.4f}  Penalized Loss: {loss_with_penalty.item():.4f}')

Loss: 0.1054  Penalized Loss: 0.1070


In [12]:
import torchvision
from torchvision import transforms

In [13]:
img_path = './'

transform = transforms.Compose(
    [transforms.ToTensor()]
)

mnist_dataset = torchvision.datasets.MNIST(root=img_path, train=True, transform=transform,
                                           download=True)
from torch.utils.data import Subset

mnist_valid_dataset = Subset(
    mnist_dataset, torch.arange(10000)
)

mnist_train_dataset = Subset(
    mnist_dataset, torch.arange(10000, len(mnist_dataset))
)

mnist_test_dataset = torchvision.datasets.MNIST(root=img_path, train=False, transform=transform, 
                                                download=False)

100%|██████████| 9.91M/9.91M [00:03<00:00, 2.99MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 303kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 2.60MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 5.54MB/s]


In [14]:
from torch.utils.data import DataLoader
batch_size = 64
torch.manual_seed(1)

train_dl = DataLoader(mnist_train_dataset, batch_size, shuffle=True)
valid_dl = DataLoader(mnist_valid_dataset, batch_size, shuffle=False)